# Solver Demo

This notebook demonstrates the production `solve_order()` API from the simplest case to full multi-carton packing. The examples are deliberately small enough to inspect by eye. No packing logic is reimplemented in the notebook.

The demo covers:

1. one item in one carton;
2. multiple items sharing one carton;
3. a geometry rejection where volume alone would be misleading;
4. an order that requires multiple cartons.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from bin_packing_3d import solve_order
from bin_packing_3d.models import UnpackableItemError

def show_result(result):
    print('Status:', result.status)
    print('Internal strategy:', result.strategy)
    print('Cartons used:', result.metrics.box_count)
    print('Validation passed:', result.validation.valid)
    for box in result.packed_boxes:
        print(f'\n{box.instance_id} -> {box.box_code}')
        for p in box.placements:
            print(' ', p.item_instance_id, 'orientation=', p.orientation, 'position=', p.position)


## 1. Single item → single carton

Start with the simplest proof: the API should select the smallest carton that can physically contain the item.


In [ ]:
boxes = [
    {'Code': 'Small', 'Length': 20, 'Width': 20, 'Height': 20, 'MaxWeight': 20},
    {'Code': 'Large', 'Length': 30, 'Width': 30, 'Height': 30, 'MaxWeight': 20},
]
order = {'OrderId': 'demo-1', 'Items': [
    {'Code': 'A', 'Length': 15, 'Width': 10, 'Height': 10, 'Weight': 1, 'Quantity': 1, 'VerticalRotation': 0}
]}

result = solve_order(order, boxes, {'mode': 'fast', 'bin_buffer': {'height': 0}}, trace=True)
show_result(result)


## 2. Multiple items → one carton

Two `15 × 10 × 10` items can coexist inside a `20 × 20 × 20` carton. This demonstrates actual XYZ coexistence rather than a volume-only check.


In [ ]:
boxes = [{'Code': 'Cube20', 'Length': 20, 'Width': 20, 'Height': 20, 'MaxWeight': 20}]
order = {'OrderId': 'demo-2', 'Items': [
    {'Code': 'A', 'Length': 15, 'Width': 10, 'Height': 10, 'Weight': 1, 'Quantity': 2, 'VerticalRotation': 0}
]}

result = solve_order(order, boxes, {'mode': 'fast', 'bin_buffer': {'height': 0}})
show_result(result)
assert result.metrics.box_count == 1
assert result.validation.valid


## 3. Geometry validation: enough volume does not mean it fits

A `30 × 10 × 20` item has less volume than a `20 × 20 × 20` carton, but no allowed axis-aligned orientation can fit because one item dimension is 30 while every carton dimension is only 20.


In [ ]:
boxes = [{'Code': 'Cube20', 'Length': 20, 'Width': 20, 'Height': 20, 'MaxWeight': 20}]
order = {'OrderId': 'demo-3', 'Items': [
    {'Code': 'Long', 'Length': 30, 'Width': 10, 'Height': 20, 'Weight': 1, 'Quantity': 1, 'VerticalRotation': 1}
]}

try:
    solve_order(order, boxes, {'mode': 'fast', 'bin_buffer': {'height': 0}}, trace=True)
except UnpackableItemError as exc:
    print('Correctly rejected:', exc)


## 4. Multiple cartons

Three `15 × 10 × 10` items cannot all share the same `20 × 20 × 20` carton under the current deterministic candidate-placement logic, so the solver must open another carton while still returning an independently validated plan.


In [ ]:
boxes = [{'Code': 'Cube20', 'Length': 20, 'Width': 20, 'Height': 20, 'MaxWeight': 20}]
order = {'OrderId': 'demo-4', 'Items': [
    {'Code': 'A', 'Length': 15, 'Width': 10, 'Height': 10, 'Weight': 1, 'Quantity': 3, 'VerticalRotation': 0}
]}

result = solve_order(order, boxes, {'mode': 'fast', 'bin_buffer': {'height': 0}})
show_result(result)
assert result.metrics.box_count > 1
assert result.validation.valid


## What this demo proves

The same production API handles single-item selection, multi-item XYZ placement, geometry rejection and multi-carton packing. Every successful plan is independently validated by the engine.

Performance comparison is intentionally separate. The benchmark notebook will compare **Historical iHub vs Fast vs Best**, including runtime and carton efficiency, without duplicating solver logic here.
